## Patient Journey - Rx
#### Data source used
* PHCDW.PHCDW_DSAA.IQV_LAAD_RX_VW
* PHCDW.PHCDW_DSAA.CKD_PRODUCT_MAPPING_VW
* PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PRODUCT
* PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PROVIDER

#### Steps:
1. Query rx claims data after 2017-01-01 for patient who visited neph after 2018-10-01
2. Get date level medication list (with 7-day additional window for gaps between refills) for each patient
3. Save table to databricks

#### Notes:
1. Treatment Cateogry and Brand

|TREATMENT_CATEGORY            |BRAND                         |
|------------------------------|------------------------------|
|ACE/ ARB                      |ACE/ ARB                      |
|Calcium channel / Beta blocker|Calcium channel / Beta blocker|
|Combo drug                    |Combo drug                    |
|Diuretic_NON_MRA              |Diuretic_NON_MRA              |
|Insulin                       |Insulin                       |
|metformin                     |metformin                     |
|MRA                           |MRA                           |
|SGLT2                         |INVOKANA 300MG                |
|SGLT2                         |FARXIGA                       |
|SGLT2                         |SGLT2                         |
|SGLT2                         |INVOKANA 100MG                |
|SGLT2                         |JARDIANCE                     |

* when PROD.PRODUCT_NAME like '%INVOKANA%' and  PRODUCT_STRENGTH = '100 MG' then 'INVOKANA 100MG'
* when PROD.PRODUCT_NAME like '%INVOKANA%' and  PRODUCT_STRENGTH = '300 MG' then 'INVOKANA 300MG'
* when PROD.PRODUCT_NAME like '%JARDIANCE%' then 'JARDIANCE'
* when PROD.PRODUCT_NAME like '%FARXIGA%' then 'FARXIGA'

## 1. Setup Snowflake connection and load data

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

### Load Libraries

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### Get Data from Scope

[REDACTED]

[REDACTED]

[REDACTED]

[REDACTED]



### Define Snowflake Connection Function
Choose appropiate below items based on the Brand
- Role
- Warehouse
- Schema
- Database

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

Run ID: 20260920
Prediction Window: 2026-10-07 → 2026-11-06
Lookback Window: 2025-08-31 → 2026-08-31


Static File Paths

Feature Accelerator Parameters For Patient Level Features

datetime.date(2026, 10, 7)

2026-10-01 2026-10-01


Rx Parameters

Px Parameters

Dx Parameters

In [0]:
%run "./00_cohort_config"

Lab Test Parameters

# Cohort Configuration

Shared config notebook for the 6 patient journey validation notebooks. 
Defines the offset, data boundaries, cohort labels (C1–C22), and the cohort assignment function.

**Usage:** Each patient journey notebook should `%run` this notebook early on.

Feature Selection Metrics

List of Products among different classes

OFFSET_DAY      = 0
DATA_START_DATE = 2023-01-01
COHORT_ORIGIN   = 2025-01-01
COHORT_LAST     = 2026-10-01


Write Input File  (HCP Level Features Path)

Date Start:  2025-08-31
Date End:  2026-08-31
Analysis Start Month:  201901
Current Month:  202608
Complete History Start Month:  201901
Complete History End Month:  202608


curr_date: 2026-10-19
run_date : 2026-09-18


## 2. Data cleanup and processing

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

In [0]:
# ===============================
# Step 1: Load / Assume tables as DataFrames
# ===============================
fact_rx_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX")
demographic_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PTNT_DMGRPHC")
provider_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR")
product_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD")
plan_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PLN")

nba_hcp_df = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/kerendia_nba_3_hcp_model_base_hcp_universe_for_inference").select('PROVIDER_ID').distinct()

ckd_mapping_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_DSAA.CKD_PRODUCT_MAPPING_VW")
print("Shape :", fact_rx_df.count(), len(fact_rx_df.columns))
print("Shape :", demographic_df.count(), len(demographic_df.columns))
print("Shape :", provider_df.count(), len(provider_df.columns))
print("Shape :", product_df.count(), len(product_df.columns))
print("Shape :", plan_df.count(), len(plan_df.columns))
print("Shape :", nba_hcp_df.count(), len(nba_hcp_df.columns))
print("Shape :", ckd_mapping_df.count(), len(ckd_mapping_df.columns))

Shape : 5967645899 60
Shape : 197723614 14
Shape : 3010168 30
Shape : 18264 26
Shape : 19260 20
Shape : 231412 1
Shape : 476 3


In [0]:
# ===============================
# Step 2: Filter Fact RX & Join Dimensions
# ===============================
rx_base = (
    fact_rx_df.alias("f")
    .join(nba_hcp_df.alias("hcp"), F.col("f.PROVIDER_ID") == F.col("hcp.PROVIDER_ID"))
    .join(provider_df.alias("p"), F.col("f.PROVIDER_ID") == F.col("p.PROVIDER_ID"))
    .join(product_df.alias("prd"), F.col("f.NDC_CD") == F.col("prd.NDC_CD"))
    .join(plan_df.alias("pl"), F.col("f.PAYER_PLAN_ID") == F.col("pl.PAYER_PLAN_ID"))
    .join(demographic_df.alias("d"), F.col("f.PATIENT_ID") == F.col("d.PATIENT_ID"), "left")
    .join(ckd_mapping_df.alias("map"), F.col("prd.PRODUCT_NAME") == F.col("map.PRODUCT_NAME"))
    .filter(
        (F.col("f.SVC_DT") >= F.lit(DATA_START_DATE)) &
        (F.col("p.NPI_NUMBER").isNotNull()) &
        (~F.lower(F.col("map.Treatment_Category")).like("%other%"))
    )
    .select(
        "f.PATIENT_ID",
        "f.SVC_DT",
        "f.DAYS_SUPPLY",
        "prd.PRODUCT_NAME",
        "prd.PRODUCT_STRENGTH",
        "map.Treatment_Category",
        "p.NPI_NUMBER",
        "p.PRIMARY_SPECIALTY_DESC"
    )
)
# Validation filter: 10 test HCPs (from config)
rx_base, test_patients = apply_hcp_filter(rx_base)

print("Shape :", rx_base.count(), len(rx_base.columns))
rx_base.limit(100).display()

Filtered to 3863 patients across 10 HCPs
Shape : 133949 8


PATIENT_ID,SVC_DT,DAYS_SUPPLY,PRODUCT_NAME,PRODUCT_STRENGTH,Treatment_Category,NPI_NUMBER,PRIMARY_SPECIALTY_DESC
2398635043,2024-08-02,30.000000,CARVEDILOL,25 MG,Calcium channel / Beta blocker,1710581061,PHYSICIAN ASSISTANT
502011727,2023-09-06,30.000000,FUROSEMIDE,40 MG,Diuretic_NON_MRA,1871876011,NEPHROLOGY
1130109818,2025-06-27,90.000000,AMLODIPINE BESYLATE,5 MG,Calcium channel / Beta blocker,1518288877,NEPHROLOGY
1243638080,2024-03-23,90.000000,LISINOPRIL,10 MG,ACE/ ARB,1790932150,NURSE PRACTITIONER
639432891,2025-12-11,7.000000,HYDROCHLOROTHIAZIDE,25 MG,Diuretic_NON_MRA,1962971952,NURSE PRACTITIONER
1453986959,2025-12-03,90.000000,AMLODIPINE BESYLATE,10 MG,Calcium channel / Beta blocker,1770843625,FAMILY MEDICINE
1416409625,2023-11-14,30.000000,LOSARTAN POTASSIUM,50 MG,Combo drug,1497050801,NURSE PRACTITIONER
1521298802,2024-06-09,90.000000,LISINOPRIL,40 MG,ACE/ ARB,1588862817,NEPHROLOGY
1359044937,2024-07-10,90.000000,AMLODIPINE BESYLATE,5 MG,Calcium channel / Beta blocker,1508365479,NURSE PRACTITIONER
387247823,2025-10-21,30.000000,LOSARTAN POTASSIUM,50 MG,Combo drug,1174670418,FAMILY MEDICINE


In [0]:
# ===============================
# Step 3: Compute END_DT per row
# ===============================
max_date = fact_rx_df.agg(F.max("SVC_DT").alias("max_date")).collect()[0]["max_date"]

rx_enriched = (
    rx_base
    .withColumn("DAYS_SUPPLY_T", F.round(F.col("DAYS_SUPPLY")).cast("int"))
    .withColumn("END_DT", F.least(
        F.expr("date_add(SVC_DT, DAYS_SUPPLY_T + 7)"),
        F.lit(max_date)
    ))
)
print("Shape :", rx_enriched.count(), len(rx_enriched.columns))
print("Max Date:", max_date)
rx_enriched.filter(F.col("PATIENT_ID") == "590252630").display()

Shape : 133949 10
Max Date: 2026-09-25


PATIENT_ID,SVC_DT,DAYS_SUPPLY,PRODUCT_NAME,PRODUCT_STRENGTH,Treatment_Category,NPI_NUMBER,PRIMARY_SPECIALTY_DESC,DAYS_SUPPLY_T,END_DT
590252630,2024-12-22,90.000000,TELMISARTAN,80 MG,ACE/ ARB,1457555021,INTERVENTIONAL CARDIOLOGY,90,2025-03-29
590252630,2024-09-25,90.000000,TELMISARTAN,80 MG,ACE/ ARB,1457555021,INTERVENTIONAL CARDIOLOGY,90,2024-12-31
590252630,2025-06-12,90.000000,TELMISARTAN,80 MG,ACE/ ARB,1457555021,INTERVENTIONAL CARDIOLOGY,90,2025-09-17
590252630,2025-03-18,90.000000,TELMISARTAN,80 MG,ACE/ ARB,1457555021,INTERVENTIONAL CARDIOLOGY,90,2025-06-23
590252630,2024-12-22,90.000000,METOPROLOL TARTRATE,100 MG,Calcium channel / Beta blocker,1457555021,INTERVENTIONAL CARDIOLOGY,90,2025-03-29
590252630,2024-09-25,90.000000,METOPROLOL TARTRATE,100 MG,Calcium channel / Beta blocker,1457555021,INTERVENTIONAL CARDIOLOGY,90,2024-12-31
590252630,2025-06-12,90.000000,METOPROLOL TARTRATE,100 MG,Calcium channel / Beta blocker,1457555021,INTERVENTIONAL CARDIOLOGY,90,2025-09-17
590252630,2025-03-18,90.000000,METOPROLOL TARTRATE,100 MG,Calcium channel / Beta blocker,1457555021,INTERVENTIONAL CARDIOLOGY,90,2025-06-23
590252630,2024-07-12,90.000000,FARXIGA,10 MG,SGLT2,1376174987,NURSE PRACTITIONER,90,2024-10-17
590252630,2024-04-10,90.000000,FARXIGA,10 MG,SGLT2,1376174987,NURSE PRACTITIONER,90,2024-07-16


In [0]:
# ===============================
# Step 4: Map Category using Spark SQL expressions (vectorized)
# ===============================
rx_enriched = rx_enriched.withColumn(
    "CATEGORY",
    F.when((F.col("PRODUCT_NAME").like("%INVOKANA%")) & (F.col("PRODUCT_STRENGTH") == "100 MG"), "INVOKANA 100MG")
     .when((F.col("PRODUCT_NAME").like("%INVOKANA%")) & (F.col("PRODUCT_STRENGTH") == "300 MG"), "INVOKANA 300MG")
     .when(F.col("PRODUCT_NAME").like("%JARDIANCE%"), "JARDIANCE")
     .when(F.col("PRODUCT_NAME").like("%FARXIGA%"), "FARXIGA")
     .when(F.col("PRODUCT_NAME").isin(
         'ALOGLIPTIN BENZOATE','ALOGLIPTIN-METFORMIN HCL','ALOGLIPTIN-PIOGLITAZONE',
         'SITAGLIPTIN-METFORMIN HCL','SITAGLIPTIN PHOSPHATE','LINAGLIPTIN-METFORMIN HCL',
         'SAXAGLIPTIN-METFORMIN HCL','SAXAGLIPTIN HCL','LINAGLIPTIN','ALOGLIPTIN',
         'ALOGLIPTIN/METFORMIN HCL','ALOGLIPTIN/PIOGLITAZONE','JANUMET','JANUMET XR',
         'JANUVIA','JENTADUETO','JENTADUETO XR','KAZANO','KOMBIGLYZE XR','NESINA',
         'ONGLYZA','OSENI','TRADJENTA'
     ), "DPP-IV")
     .when(F.col("PRODUCT_NAME").isin(
         'LIXISENATIDE','EXENATIDE','SEMAGLUTIDE','ALBIGLUTIDE','DULAGLUTIDE',
         'LIRAGLUTIDE','INSULIN DEGLUDEC-LIRAGLUTIDE','ADLYXIN','ADLYXIN STARTER PACK',
         'BYDUREON','BYDUREON BCISE','BYDUREON PEN','BYETTA','OZEMPIC','RYBELSUS',
         'TANZEUM','TRULICITY','VICTOZA','XULTOPHY 100/3.6'
     ), "GLP-1")
     .otherwise(F.col("Treatment_Category"))
)
print("Shape :", rx_enriched.count(), len(rx_enriched.columns))
rx_enriched.filter(F.col("PATIENT_ID") == "590252630").display()

Shape : 133949 11


PATIENT_ID,SVC_DT,DAYS_SUPPLY,PRODUCT_NAME,PRODUCT_STRENGTH,Treatment_Category,NPI_NUMBER,PRIMARY_SPECIALTY_DESC,DAYS_SUPPLY_T,END_DT,CATEGORY
590252630,2024-12-22,90.000000,TELMISARTAN,80 MG,ACE/ ARB,1457555021,INTERVENTIONAL CARDIOLOGY,90,2025-03-29,ACE/ ARB
590252630,2024-09-25,90.000000,TELMISARTAN,80 MG,ACE/ ARB,1457555021,INTERVENTIONAL CARDIOLOGY,90,2024-12-31,ACE/ ARB
590252630,2025-06-12,90.000000,TELMISARTAN,80 MG,ACE/ ARB,1457555021,INTERVENTIONAL CARDIOLOGY,90,2025-09-17,ACE/ ARB
590252630,2025-03-18,90.000000,TELMISARTAN,80 MG,ACE/ ARB,1457555021,INTERVENTIONAL CARDIOLOGY,90,2025-06-23,ACE/ ARB
590252630,2024-12-22,90.000000,METOPROLOL TARTRATE,100 MG,Calcium channel / Beta blocker,1457555021,INTERVENTIONAL CARDIOLOGY,90,2025-03-29,Calcium channel / Beta blocker
590252630,2024-09-25,90.000000,METOPROLOL TARTRATE,100 MG,Calcium channel / Beta blocker,1457555021,INTERVENTIONAL CARDIOLOGY,90,2024-12-31,Calcium channel / Beta blocker
590252630,2025-06-12,90.000000,METOPROLOL TARTRATE,100 MG,Calcium channel / Beta blocker,1457555021,INTERVENTIONAL CARDIOLOGY,90,2025-09-17,Calcium channel / Beta blocker
590252630,2025-03-18,90.000000,METOPROLOL TARTRATE,100 MG,Calcium channel / Beta blocker,1457555021,INTERVENTIONAL CARDIOLOGY,90,2025-06-23,Calcium channel / Beta blocker
590252630,2024-07-12,90.000000,FARXIGA,10 MG,SGLT2,1376174987,NURSE PRACTITIONER,90,2024-10-17,FARXIGA
590252630,2023-08-18,90.000000,METFORMIN HYDROCHLORIDE E,500 MG/24HR,metformin,1508230566,NURSE PRACTITIONER,90,2023-11-23,metformin


In [0]:
# ===============================
# Step 5: Expand dates per day using explode(sequence())
# ===============================
rx_expanded = rx_enriched.withColumn(
    "TXN_DATE",
    F.explode(F.expr("sequence(SVC_DT, END_DT, interval 1 day)"))
)

# Convert to cohort-month granularity (using offset from config)
rx_expanded = rx_expanded.withColumn(
    "COHORT_MONTH",
    F.date_format(assign_cohort_month(F.col("TXN_DATE")), "yyyy-MM-dd")
)
print("Shape :", rx_expanded.count(), len(rx_expanded.columns))
rx_expanded.filter(F.col("PATIENT_ID") == "590252630").orderBy("COHORT_MONTH").display()

Shape : 9357986 13


PATIENT_ID,SVC_DT,DAYS_SUPPLY,PRODUCT_NAME,PRODUCT_STRENGTH,Treatment_Category,NPI_NUMBER,PRIMARY_SPECIALTY_DESC,DAYS_SUPPLY_T,END_DT,CATEGORY,TXN_DATE,COHORT_MONTH
590252630,2023-01-04,90.000000,TELMISARTAN,80 MG,ACE/ ARB,1508230566,NURSE PRACTITIONER,90,2023-04-11,ACE/ ARB,2023-01-28,2023-01-01
590252630,2023-01-04,90.000000,TELMISARTAN,80 MG,ACE/ ARB,1508230566,NURSE PRACTITIONER,90,2023-04-11,ACE/ ARB,2023-01-31,2023-01-01
590252630,2023-01-28,90.000000,METFORMIN HYDROCHLORIDE E,500 MG/24HR,metformin,1508230566,NURSE PRACTITIONER,90,2023-05-05,metformin,2023-01-29,2023-01-01
590252630,2023-01-28,90.000000,METFORMIN HYDROCHLORIDE E,500 MG/24HR,metformin,1508230566,NURSE PRACTITIONER,90,2023-05-05,metformin,2023-01-30,2023-01-01
590252630,2023-01-04,90.000000,TELMISARTAN,80 MG,ACE/ ARB,1508230566,NURSE PRACTITIONER,90,2023-04-11,ACE/ ARB,2023-01-27,2023-01-01
590252630,2023-01-02,30.000000,FUROSEMIDE,20 MG,Diuretic_NON_MRA,1144227752,INTERVENTIONAL CARDIOLOGY,30,2023-02-08,Diuretic_NON_MRA,2023-01-03,2023-01-01
590252630,2023-01-28,90.000000,METFORMIN HYDROCHLORIDE E,500 MG/24HR,metformin,1508230566,NURSE PRACTITIONER,90,2023-05-05,metformin,2023-01-31,2023-01-01
590252630,2023-01-02,30.000000,FUROSEMIDE,20 MG,Diuretic_NON_MRA,1144227752,INTERVENTIONAL CARDIOLOGY,30,2023-02-08,Diuretic_NON_MRA,2023-01-14,2023-01-01
590252630,2023-01-04,90.000000,TELMISARTAN,80 MG,ACE/ ARB,1508230566,NURSE PRACTITIONER,90,2023-04-11,ACE/ ARB,2023-01-21,2023-01-01
590252630,2023-01-04,90.000000,TELMISARTAN,80 MG,ACE/ ARB,1508230566,NURSE PRACTITIONER,90,2023-04-11,ACE/ ARB,2023-01-06,2023-01-01


In [0]:
# ===============================
# Step 6: Aggregate per patient per month
# ===============================
final_df = (
    rx_expanded
    .groupBy("PATIENT_ID", "COHORT_MONTH")
    .agg(
        F.collect_set("Treatment_Category").alias("TREATMENT_CATEGORY"),
        F.collect_set("CATEGORY").alias("CATEGORY")
    )
    .orderBy("PATIENT_ID", "COHORT_MONTH")
)

# Join cohort labels (C1-C22); rows before C1 get null label (lookback)
final_df = final_df.join(
    df_cohort_ref.select(
        F.date_format("cohort_ref_month", "yyyy-MM-dd").alias("_ref_month_str"),
        "cohort_label"
    ),
    final_df.COHORT_MONTH == F.col("_ref_month_str"),
    "left"
).drop("_ref_month_str")

print("Shape :", final_df.count(), len(final_df.columns))
final_df.filter(F.col("PATIENT_ID") == "590252630").orderBy("COHORT_MONTH").display()


Shape : 113587 5


PATIENT_ID,COHORT_MONTH,TREATMENT_CATEGORY,CATEGORY,cohort_label
590252630,2023-01-01,"List(Diuretic_NON_MRA, Calcium channel / Beta blocker, metformin , ACE/ ARB)","List(Diuretic_NON_MRA, metformin , Calcium channel / Beta blocker, ACE/ ARB)",null
590252630,2023-02-01,"List(Diuretic_NON_MRA, metformin , Calcium channel / Beta blocker, ACE/ ARB)","List(Diuretic_NON_MRA, metformin , ACE/ ARB, Calcium channel / Beta blocker)",null
590252630,2023-03-01,"List(Calcium channel / Beta blocker, ACE/ ARB, metformin )","List(Calcium channel / Beta blocker, metformin , ACE/ ARB)",null
590252630,2023-04-01,"List(ACE/ ARB, metformin , Calcium channel / Beta blocker)","List(metformin , ACE/ ARB, Calcium channel / Beta blocker)",null
590252630,2023-05-01,"List(ACE/ ARB, Calcium channel / Beta blocker, metformin )","List(ACE/ ARB, Calcium channel / Beta blocker, metformin )",null
590252630,2023-06-01,"List(ACE/ ARB, metformin )","List(metformin , ACE/ ARB)",null
590252630,2023-07-01,"List(ACE/ ARB, metformin )","List(ACE/ ARB, metformin )",null
590252630,2023-08-01,"List(ACE/ ARB, metformin )","List(ACE/ ARB, metformin )",null
590252630,2023-09-01,"List(ACE/ ARB, metformin )","List(metformin , ACE/ ARB)",null
590252630,2023-10-01,"List(metformin , ACE/ ARB)","List(ACE/ ARB, metformin )",null


In [0]:
df_rx_clean = final_df

In [0]:
df_rx_clean = df_rx_clean.withColumnRenamed("COHORT_MONTH", "DATE")

In [0]:
# DATE is already in yyyy-MM-dd format from assign_cohort_month
df_rx_clean = df_rx_clean.withColumn("DATE", F.to_date(F.col("DATE"), "yyyy-MM-dd"))

In [0]:
print("Shape :", df_rx_clean.count(), len(df_rx_clean.columns))

Shape : 113587 5


In [0]:
# VALIDATION: write to a separate path so we don't overwrite production data
df_rx_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_rx_journey_validation")